# ObsAstro26 P2, Spectral Fitting for Supernovae

Sharon Xuesong Wang \
refreshed 2026/5/13\
refreshed 2025/5/7\
updated 2023/5/15\
created 2023/5/10

**Goals:**
- Understand model fitting and perform a bare basic data analysis.
- Fit one spectral line with gaussian, and estimate the error bar on the redshift.


In [ ]:
# python modules that we will use
import os
import numpy as np
from astropy.io import fits
from astropy.stats import sigma_clip
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from pathlib import Path
import specutils

%matplotlib inline
import matplotlib.pylab as plt

In [ ]:
# change plotting defaults
plt.rc('axes', labelsize=14)
plt.rc('axes', labelweight='bold')
plt.rc('axes', titlesize=16)
plt.rc('axes', titleweight='bold')
plt.rc('font', family='sans-serif')
plt.rcParams['figure.figsize'] = (17, 7)

# Part I, fitting a simple Gaussian line using the demo data from lecture 10

In [ ]:
from astropy.io import ascii

target_name='SN2026kid'
data_dir = f'../data/{target_name}'
spec_file=list(Path(data_dir).glob('*.txt'))
spec_data = ascii.read(spec_file[0])
wavs = spec_data['col1']
spec = spec_data['col2']

In [ ]:
# this is repeating what you've done, to do a first guess
# Now plot the extracted 1-D spectrum with the x-axis being wavelengths in angstroms
plt.figure()
plt.plot(wavs, spec)

# Zoom in on the brightest emission line - that's the H alpha emission line
plt.xlim(6000, 7000)
plt.ylim(1e-15,3e-15)
plt.grid()

# make a guess at the central wavelength
obswav_guess = 6570
plt.axvline(obswav_guess, ls='dashed', c='k')
plt.show()

In [ ]:
# Define a gaussian to fit the line
# use a Gaussian function to model spectral lines
def gaussian(x, *params):
    amp, x0, sigma = params
    gauss = amp * np.exp(-(x - x0) ** 2 / (2 * sigma ** 2))
    return gauss

In [ ]:
# fit a Gaussian to find line center
guess = 2.7e-15, obswav_guess, 1
bounds = ((0, obswav_guess - 25, 0), (np.inf, obswav_guess + 25, np.inf))
popt, pcov = curve_fit(gaussian, wavs, spec, p0=guess, bounds=bounds)

obswav = popt[1]
obswav_error = np.sqrt(pcov[1, 1])
print(f'Observed wavelength of H-alpha is {obswav:.3f} +/- {obswav_error:.3f}')

In [ ]:
# check out your fit!
line_fitted = gaussian(wavs, *popt)
plt.plot(wavs, spec)
plt.plot(wavs, line_fitted,'--')
plt.xlim(6500, 6700)
plt.grid()
plt.show()

In [ ]:
# use the known laboratory wavelength to determine the redshift
restwav = 6562.78
redshift = obswav / restwav - 1

Suppose you know the error in the wavelength solution is 0.17 angstrom, and it's independent to your wavelength center measurement error, which can be derived using the diagonal terms in pcov (they are the variances for each parameter).

How would you estimate the redshift error?

In [ ]:
# adding errors?
redshift_error = np.sqrt(0.17**2+obswav_error**2) / restwav
print(f'The redshift is {redshift:.5f} +/- {redshift_error:.5f}')

## Part II, Fitting the H $\alpha$ line with more noise and a continuum

Using data from ObsAstro23, let's tackle a more challenging case with noise and continuum!

The fit file contains the confirmation image where it shows how the slit is placed on the galaxy (just for fun).

The two text files contain the extracted, flux calibrated spectrum (erg/cm2/s/A) on 5/2 and 5/12 in 2023.

In [ ]:
# First, read the spectra in
data_1 = np.loadtxt(spec_file[2])
wav1 = data_1[:,0]
flux1 = data_1[:,1]
# We are mostly going to work with the data from 0502 for the modeling, 
# but you can fit 0512 spectrum too if you want. :)

data_2 = np.loadtxt(spec_file[1])
wav2 = data_2[:,0]
flux2 = data_2[:,1]

In [ ]:
# Let's plot them out to take a look!
plt.figure()
plt.plot(wav1, flux1)
plt.plot(wav2, flux2)
plt.legend(["1", "2"])
plt.xlabel(r'Wavelength ($\AA$)')
plt.ylabel(r'Flux ($erg/cm^2/s/\AA$)')
plt.grid()
plt.show()

In [ ]:
# Now, the small emission line near 6700 angstroms is the Halpha line. 
# Let's zoom in to use just this portion of the spectrum near Halpha.
# Try to stay clean out of other spectral lines, 
# but you want a large enough wavelength range for good continuum fitting!
left = 6350  # left border
right = 7000  # right border
fit_wav1 = wav1[ (wav1 > left) & (wav1 < right) ]
fit_flux1 = flux1[ (wav1 > left) & (wav1 < right) ]
fit_wav2 = wav2[ (wav2 > left) & (wav2 < right) ]
fit_flux2 = flux2[ (wav2 > left) & (wav2 < right) ]

# Plot this zoom-in portion
plt.figure()
plt.plot(fit_wav1, fit_flux1)
plt.plot(fit_wav2, fit_flux2)
plt.show()

In [ ]:
# Ok, next, we want to fit the contiuum, because there's clearly a continuum slope...
# Let's define a region to mask out, which contains the Halpha line.
wav_min = 6500
wav_max = 6650

# Remove the data points within the specified range
mask = (fit_wav1 < wav_min) | (fit_wav1 > wav_max)
wav1_filtered = fit_wav1[mask]
flux1_filtered = fit_flux1[mask]

In [ ]:
# Let's fit a straight line to the masked data using linear regression to get a continuum model
from scipy.stats import linregress
slope, intercept, r_value, p_value, std_err = linregress(wav1_filtered, flux1_filtered)

In [ ]:
# Plot the original data, filtered or masked data, and the fitted line as the continuum model
plt.figure()
plt.scatter(fit_wav1, fit_flux1, label='Original Data')
plt.scatter(wav1_filtered, flux1_filtered, label='Filtered Data', color='orange')

# Use intercept and slope to get the model for the continuum
cont_model = slope * fit_wav1 + intercept

plt.plot(fit_wav1, cont_model, 'r', label='Fitted Line')
plt.xlabel(r'Wavelength ($\AA$)')
plt.ylabel(r'Flux ($erg/cm^2/s/\AA$)')
plt.xlim((left,right))
plt.legend()
plt.show()

In [ ]:
# Let's normalize the spectrum and check it out!
norm_flux1 = fit_flux1 / cont_model
plt.plot(fit_wav1, norm_flux1)
plt.xlabel(r'Wavelength ($\AA$)')
plt.ylabel('Normalized Flux')
plt.show()

In [ ]:
# We need to re-define a new gaussian, because now you have an offset, 
# which means the gaussian does not go to zero at infinity but goes to this offset.
def gaussian(x, amp, x0, sigma, offset):
    gauss = amp * np.exp(-0.5 * ((x - x0) / sigma) ** 2) + offset
    return gauss

In [ ]:
# Let's fit a gaussian to the normalized spectrum containing the Halpha line!
shifted_guess = 6573  # make an initial guess of the wavelength of the Halpha line in the galaxy frame

# Think: What's a good guess for the offset?
guess1 = 0.2, shifted_guess, 5, 1
bounds1 = ((-np.inf, shifted_guess - 50, 0, -np.inf), (np.inf, shifted_guess + 50, np.inf, np.inf))
popt1, pcov1 = curve_fit(gaussian, fit_wav1, norm_flux1, p0 = guess1, bounds = bounds1)

In [ ]:
# Now let's plot the data and the best-fit gaussian model!
line_fitted1 = gaussian(fit_wav1, *popt1)
plt.plot(fit_wav1, norm_flux1, alpha = 0.5)
plt.plot(fit_wav1, line_fitted1, 'b--')
plt.axvline(shifted_guess, ls='dashed', c='k')
# plt.xlim(6680, 6750)
plt.show()

In [ ]:
# What's the measured redshift and its error bar (ignore error from wavelength calibration)?
rest_wave = 6562.8  # rest-frame wavelength of H alpha
redshift = (popt1[1] - rest_wave) / rest_wave
redshift_error = np.sqrt(0.17**2+np.sqrt(pcov1[1, 1])**2) / rest_wave
print(f'The redshift is {redshift:.5f} +/- {redshift_error:.5f}')

## Extra questions to explore:

Now, there are additional lines besides the Halpha line that belongs to the host galaxy. \
You can tell what they are using the figure in this folder labeling the lines, or this tool you've used here:\
http://astronomy.nmsu.edu/geas/labs/spectrum_fitter03/html5/gal_spectra.html

Pick a line that you like, identify it, and then measure its equivalent widths in each of the two nights using the package *specutils*. You can see a tutorial on how to measure EW here:\
https://specutils.readthedocs.io/en/stable/analysis.html

Remember to normalize the spectrum first before you measure the EW!

pypeit